# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a guide for loading, exploring, and analyzing the FAIR² colorectal cancer survivors dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library and the Croissant metadata schema.

### Dataset Source
The dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview

Let's review the record sets, their fields, and column IDs available in the dataset. All Croissant entities (record sets, fields, columns) are referenced using their `@id`.

We use the dataset's `record_sets` to examine structure.

In [ ]:
from pprint import pprint

print("Available record sets:")
for rs in dataset.metadata.record_sets:
    print(f"  Record set name: {rs.name}\n    @id: {rs.id}")
    print("    Fields:")
    for f in rs.fields:
        print(f"      Field name: {f.name} (@id: {f.id})  -- type: {f.data_type}")
        if hasattr(f, 'column'):
            # In Croissant 1.0, f.column can be present for tabular columns
            if isinstance(f.column, list):
                for c in f.column:
                    print(f"        > column @id: {c.id} (header: {c.header})")
            elif f.column is not None:
                print(f"        > column @id: {f.column.id} (header: {f.column.header})")
    print()

## 3. Data Extraction

Let's load all data from each record set into separate pandas DataFrames. To reference these programmatically, we build a list of record set `@id`s.

We then display the columns available in one of the record sets as an example.

In [ ]:
# Gather record set @id's
record_sets = [rs.id for rs in dataset.metadata.record_sets]

dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

if dataframes:
    # Take the first available record set with data
    example_rs_id = list(dataframes.keys())[0]
    print(f'Columns in record set {example_rs_id}:')
    print(dataframes[example_rs_id].columns.tolist())
    dataframes[example_rs_id].head()
else:
    print('No records extracted. Check dataset or record set IDs.')

## 4. Exploratory Data Analysis (EDA)

For demonstration, we'll select a numeric field from the example record set, filter records, normalize the field, and group by a categorical field. All field names are referenced by `@id`. Please adjust the field IDs as needed for your data exploration.

In [ ]:
# We'll try to programmatically find a numeric field in the example record set.
import numpy as np

example_rs = None
for rs in dataset.metadata.record_sets:
    if rs.id == example_rs_id:
        example_rs = rs
        break

# Try to auto-select a first numeric and a categorical field
numeric_field_id = None
category_field_id = None
for field in example_rs.fields:
    if field.data_type in ('Integer', 'Float', 'Number') and numeric_field_id is None:
        numeric_field_id = field.id
    if field.data_type in ('Text', 'String', 'Category') and category_field_id is None:
        category_field_id = field.id

print(f"Selected numeric field @id: {numeric_field_id}")
print(f"Selected group/categorical field @id: {category_field_id}")

# If there's no data, skip processing
df = dataframes[example_rs_id]
if numeric_field_id and numeric_field_id in df.columns:
    # Remove NaNs
    numeric_values = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = numeric_values.mean() if np.isfinite(numeric_values.mean()) else 10
    filtered_df = df[numeric_values > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (numeric_values - numeric_values.mean()) / numeric_values.std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by categorical
    if category_field_id and category_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(category_field_id)[numeric_field_id].mean()
        print(f"Grouped mean of {numeric_field_id} by {category_field_id}:")
        display(grouped_df)
else:
    print("No numeric field available for analysis.")

## 5. Visualization

Visualize distributions and relationships for the selected numeric and group fields, using their Croissant `@id` references.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=20, kde=True)
    plt.title(f"Distribution of Numeric Field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    if category_field_id and category_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=df[category_field_id], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
        plt.title(f"{numeric_field_id} by {category_field_id}")
        plt.xlabel(category_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=60)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion

This notebook demonstrated how to load and explore a FAIR² tabular dataset using the Croissant schema and the `mlcroissant` library.

* We inspected available record sets, fields, and columns using their `@id` values, ensuring programmatic referencing throughout.
* We loaded tabular data into pandas DataFrames, chose numeric and group fields by `@id`, and performed basic filtering, normalization, grouping, and visualization.
* Use this template to further process, clean, or model the data as needed for downstream biomedical or clinical analysis tasks.